# C1 · 1. SEP-28k: download, clips and speaker-separated splits

Downloads SEP-28k's podcast audio, cuts the 28,177 labelled 3-second clips, and attaches the **speaker-separated SEP-28k-E splits** (Bayerl et al., 2022), so no speaker is in both training and test.

- About **32 GB** of episode audio plus 2.6 GB of clips; downloads resume, so the notebook can be re-run after a break.
- About a quarter of the podcast links are dead (checked 11 October 2026); they are logged in `download_log.csv` and their clips are skipped.
- FluencyBank's part of SEP-28k needs a TalkBank login and is not downloaded here.
- Licence: CC BY-NC 4.0, research only; cite Lea et al. (ICASSP 2021) and Bayerl et al. (TSD 2022). The audio must not be redistributed.

Settings (environment variables): `SWARA_C1_DATA` (where everything goes, default `data/sep28k`), `SWARA_C1_SHOWS` (comma-separated show codes, to download only some shows first), `SWARA_C1_WORKERS` (parallel downloads, default 4).

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # the speech service's own code (app/, training/)
DATA = Path(os.environ.get("SWARA_C1_DATA", ROOT / "data" / "sep28k"))

SHOWS = [s for s in os.environ.get("SWARA_C1_SHOWS", "").split(",") if s]
WORKERS = int(os.environ.get("SWARA_C1_WORKERS", 4))
from training import sep28k
print("data folder:", DATA)

## 1. Labels, episodes and splits (small CSVs)

In [ ]:
meta = sep28k.fetch_metadata(DATA)
episodes = sep28k.read_episodes(meta["SEP-28k_episodes.csv"])
clips = sep28k.read_clips(meta["SEP-28k-Extended_clips.csv"])
if SHOWS:
    episodes = episodes[episodes["Show"].isin(SHOWS)]
    clips = clips[clips["Show"].isin(SHOWS)]
print(f"{len(episodes)} episodes, {len(clips)} labelled clips, shows: {sorted(clips['Show'].unique())}")
clips[["Show", "EpId", "ClipId", "SEP28k-E", "Block", "Prolongation", "SoundRep", "WordRep", "Interjection"]].head()

## 2. Download and convert the episodes (resumable)

In [ ]:
t0 = time.time()
log = sep28k.download_episodes(episodes, DATA, workers=WORKERS)
print(f"{(log['status'] == 'ok').sum()} of {len(log)} episodes available ({time.time() - t0:.0f} s)")
log[log["status"] != "ok"].head(10)

## 3. Cut the 3-second clips

In [ ]:
clips = sep28k.extract_clips(clips, DATA)
index = clips[clips["path"].notna()].copy()
index.to_csv(DATA / "clips_index.csv", index=False)
print("index:", DATA / "clips_index.csv")

## 4. What is available, per split

A label counts as present when at least 2 of the 3 annotators chose it. Clips most annotators marked as poor audio, music, no speech, hard to understand or unsure are left out of training and testing (notebook 3).

In [ ]:
keep = sep28k.usable(index)
print(f"{keep.sum()} of {len(index)} downloaded clips are usable")
sep28k.summary(index[keep])